# Imports

In [1]:
## conda env: stereo_visionn
import os
import cv2
import glob
import numpy as np
from rtmlib import draw_skeleton, draw_bbox
from Util.util import *
import time
import json
from IPython.display import display, clear_output

import pyzed.sl as sl
import matplotlib.pyplot as plt

with open('./Util/models.json') as f:
    models = json.load(f)

with open("./Util/gait_analysis_properties.json", "r") as json_file:
    gait_analysis_properties = json.load(json_file)
    stereo_analysis_properties = gait_analysis_properties["stereo"]
    qualysis_analysis_properties = gait_analysis_properties["qualisys"]


# %matplotlib ipympl
# %matplotlib notebook
# from ipywidgets import interact, interactive, fixed, interact_manual
# import ipywidgets as widgets

# 2D Keypoints

## Model setup

In [ ]:
device = "cuda"  # cpu, cuda, mps
backend = "onnxruntime"  # opencv, onnxruntime, openvino
detector_name = 'YOLOX_nano' # 'YOLOX_l_COCO','YOLOX_nano','YOLOX_tiny','YOLOX_s','YOLOX_m','YOLOX_l','YOLOX_x'
pose_name = 'RTMPose_x' # (26) 'RTMPose_t', 'RTMPose_s', 'RTMPose_m', 'RTMPose_l', 'RTMPose_m2', 'RTMPose_l2', 'RTMPose_x', (133) 'RTMW_l', 'RTMW_x'
kpt_labels = models['pose_models']["26"]["kpt_labels"]

custom = Custom(det_class='YOLOX',#'RTMDet',
                det=models['detectors'][detector_name]['path'],
                det_input_size=models['detectors'][detector_name]['input_size'],
                pose_class='RTMPose',
                pose=models['pose_models']["26"]["models"][pose_name]['path'],
                pose_input_size=models['pose_models']["26"]["models"][pose_name]['input_size'],
                backend=backend,
                device=device) 

## 2D inference

In [ ]:
input_folder = ".\\stereo_videos\\validation_test"

for g in glob.glob(os.path.join(input_folder, "*.avi")):
    print(f"processing: {g}")

    keypoints_2d = []
    confidence_2d = []
    color_2d = []
    
    cap = cv2.VideoCapture(g)
    if cap.isOpened() == False:
        print("Error opening video file")

    total_num_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    processed_num_frames = 0
    start_time = time.time_ns()

    # Read until video is completed
    while cap.isOpened():

        # Capture frame-by-frame
        ret, frame = cap.read()

        if ret == True and frame is not None:
            processed_num_frames += 1
            width = frame.shape[1]
            height = frame.shape[0]

            # inference
            keypoints, scores = custom(frame)

            # visualize
            boxes = [pose_to_bbox(x) for x in keypoints]
            img_show = draw_bbox(frame, boxes, (0, 0, 255))
            # img_show = draw_skeleton(img_show, keypoints, kpt_labels)
            for kpt in keypoints[0,:,:]:
                cv2.circle(img_show, (int(kpt[0]), int(kpt[1])), 5, (0, 255, 0), 3)

            cv2.imshow("Image", cv2.resize(img_show, (int(width / 2), int(height / 2))))

            ## store results (frames * kpt * (x,y))
            # remove multiperson cases
            if keypoints.shape != (1, 26, 2):
                keypoints = keypoints[0, :, :]
                scores = scores[0, :]

            keypoints = np.squeeze(keypoints)
            color = np.array([frame[(int(x),int(y))] for x,y in keypoints[:,:2]])
            
            keypoints_2d.append(keypoints)
            confidence_2d.append(np.squeeze(scores))
            color_2d.append(color)

            # display how much time elapsed/is left
            clear_output(wait=True)
            end_time = time.time_ns()
            elapsed_time = (end_time - start_time) / 1000000000

            if int(elapsed_time % 10) == 0:

                fps = round(processed_num_frames / elapsed_time, 2)
                done_ratio = processed_num_frames / total_num_frames
                expected_duration_min = elapsed_time / (done_ratio * 60)

                display(
                    f"exp_duration: {round(expected_duration_min,2)} minutes, elapsed: {round(elapsed_time/60,2)} minutes"
                )
                display(f"done: {round(done_ratio*100,2)}%, fps: {fps}")

            # Press Q on keyboard to exit
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break

        else:
            break

    cap.release()
    cv2.destroyAllWindows()

    keypoints_2d = np.asarray(keypoints_2d)
    confidence_2d = np.asarray(confidence_2d)
    color_2d = np.asarray(color_2d)

    print(f"shape of keypoints_2d:  {keypoints_2d.shape} (frames * kpt * (x, y))")
    print(f"shape of confidence_2d: {confidence_2d.shape} (frames * kpt)")
    print(f"shape of color_2d:      {color_2d.shape} (frames * kpt * rgb)")

    

## Create new keypoint (mid_heel)

In [ ]:
left_heel = keypoints_over_time[:,24,:]
right_heel = keypoints_over_time[:,25,:]

# new keypoints between ankles 
# note: confidence is also the avg of ankles
mid_heel = (left_heel + right_heel)/2

# add the new keypoint to the rest
keypoints_over_time = np.concat((keypoints_over_time, np.expand_dims(mid_heel, axis=1)), axis=1)

kpt_labels = np.append(kpt_labels, "mid_heel")
print(f'keypoints shape (w new one added): {keypoints_over_time.shape}')

## Filter data 
(so it's less jiggly for depth)

In [ ]:
for idx, kpt in enumerate(keypoints_2d):
    print(idx, kpt.shape)
    break

In [ ]:
filtered_2d_keypoints = []

for keypoint_id, keypoint in enumerate(keypoints_2d.transpose(1,0,2)):
    result = filter_2d_keypoint(keypoint,6,7, 60, False,f'{keypoint_id}')
    filtered_2d_keypoints.append(result)

filtered_2d_keypoints = np.array(filtered_2d_keypoints).transpose(1,0,2)
print("Filtered data shape:", filtered_2d_keypoints.shape)

In [ ]:
keypoints_2d.transpose(1,0,2).shape

## Calculate scale

In [ ]:
# participant height in meters
HEIGHT_METERS = 1.8

# y coords of the top_head and mid_heel keypoints
top_head_y = keypoints_over_time[:,17,1]
mid_heel_y = mid_heel[:,1]

# participant height in pixels
height_px = (mid_heel_y - top_head_y)

# how many meters one pixel is in each frame
scale = HEIGHT_METERS/height_px

print(f"scale shape: {scale.shape}")
# scale[:5]

## Save data

In [ ]:
out_file_name = os.path.join(input_folder, f"{os.path.basename(g).split('.')[0]}.npz")

with open(out_file_name, "w") as f:
    np.savez(
        out_file_name,
        keypoints_2d=filtered_2d_keypoints,
        confidence_2d=confidence_2d,
        color=color_2d,
        kpt_labels=kpt_labels
    )

print(f"Saved 2d keypoint to {out_file_name}")

In [ ]:
kpt = 0
filter_2d_keypoint(keypoints_over_time[:,kpt,:],6,15,60,True, f"{kpt}")

In [ ]:
# bandpass
from scipy.signal import butter, filtfilt

x, y, d = keypoints_over_time[:, 0, :].T

lowcut = 1  # cut fr for lowpass
# highpass = 5 # cut fr for highpass
bandpass = [0.5, 5]
b, a = butter(2, bandpass, fs=60, btype="band", analog=False)
fx = filtfilt(b, a, x)
fy = filtfilt(b, a, y)

plt.close('all')
plt.figure(figsize=(13,5))
plt.subplot(211)
plt.plot(x, 'b', alpha=0.5)
plt.plot(fx+np.min(x),'r', alpha=0.5)
plt.title('x')


plt.subplot(212)
plt.plot(y, 'b', alpha=0.5)
plt.plot(fy + np.min(y), 'r', alpha=0.5)
plt.title('y')

plt.tight_layout()
plt.legend()

## Visualization

In [ ]:
input_vid = ".\\stereo_videos\\validation_test\\43916681.avi"

vid_basename = os.path.basename(input_vid)
folder_name = os.path.dirname(input_vid)
npz_basename = f"{vid_basename.split(".")[0]}.npz"

load_path = os.path.join(folder_name, npz_basename)
loaded_data = np.load(load_path)
print(f"loaded data : {load_path}\nwith keys: {list(loaded_data.keys())}")

keypoints = loaded_data["keypoints_2d"]
# keypoints = loaded_data["raw_keypoints"]
kpt_labels = loaded_data["kpt_labels"]
scale = loaded_data["scale"]



cap = cv2.VideoCapture(input_vid)
if cap.isOpened() == False:
    print("Error opening video file")

total_num_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
frame_idx = 0

# Read until video is completed
while cap.isOpened():

    # Capture frame-by-frame
    ret, frame = cap.read()

    if ret == True and frame is not None:
        frame_idx += 1
        width = frame.shape[1]
        height = frame.shape[0]


        for kpt, conf in zip(keypoints[frame_idx,:,:2],keypoints[frame_idx,:,2]):
            cv2.circle(frame,(int(kpt[0]),int(kpt[1])),radius=3, color=(0,0,255), thickness=2 )
            # cv2.putText(frame, f"{conf:.2f}",(int(kpt[0]),int(kpt[1])),cv2.FONT_HERSHEY_SIMPLEX,fontScale=0.75,color = (255, 255, 255))
            

        cv2.imshow("Image", cv2.resize(frame, (int(width / 2), int(height / 2))))
        time.sleep(0.01)

        # Press Q on keyboard to exit
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    else:
        break

cap.release()
cv2.destroyAllWindows()
    

# 3D Keypoints

## Load 2D data

In [ ]:
npz_load_path = "stereo_videos\\validation_test\\43916681.npz"
loaded_data = np.load(npz_load_path)

svo_path = f"{os.path.splitext(npz_load_path)[0]}.svo2"

keypoints_2d = loaded_data["keypoints_2d"] # TODO: if the whole piplenie runs at onece, this has to be changed to filtered_keypoints_2d
confidence_2d = loaded_data["confidence_2d"]
color = loaded_data["color"]
kpt_labels = loaded_data["kpt_labels"]

print(f"loaded data : {npz_load_path}\nwith keys: {list(loaded_data.keys())}")
print(f"2d keypoint shape: {keypoints_2d.shape}")


## Get depth from StereoLabs 

In [ ]:
# TODO: update ZED sdk, try it with latest models, methods

keypoints_3d = []
keypoints_point_cloud = []

# Create a ZED camera object
zed = sl.Camera()

input_type = sl.InputType()
input_type.set_from_svo_file(svo_path)  # Set init parameter to run from the .svo
init_parameters = sl.InitParameters(input_t=input_type, svo_real_time_mode=False)

init_parameters.depth_mode = sl.DEPTH_MODE.NEURAL_PLUS
init_parameters.coordinate_units = sl.UNIT.METER  # CENTIMETER, METER, MILLIMETER
# init_parameters.coordinate_system = sl.COORDINATE_SYSTEM.RIGHT_HANDED_Y_UP

# TODO: see how own filtering does without this built in one
init_parameters.depth_stabilization = 30 # 0 to trun it off, otherwise 1-100 linear. default is 30

# Open the ZED
err = zed.open(init_parameters)
framerate = zed.get_camera_information().camera_configuration.fps
resolution = zed.get_camera_information().camera_configuration.resolution

svo_depth = sl.Mat()
svo_image = sl.Mat()
# svo_confidance = sl.Mat()
svo_xyz = sl.Mat()

frame_idx = 0

while frame_idx < keypoints_2d.shape[0] - 1 and zed.grab() == sl.ERROR_CODE.SUCCESS:

    current_keypoints = []
    current_xyz = []

    # Get frame count
    frame_idx = zed.get_svo_position()

    # get color image and depth map
    zed.retrieve_measure(svo_depth, sl.MEASURE.DEPTH)
    zed.retrieve_image(svo_image, sl.VIEW.LEFT) # BGRA image
    # zed.retrieve_measure(svo_confidance, sl.MEASURE.CONFIDENCE)
    zed.retrieve_measure(svo_xyz, sl.MEASURE.XYZRGBA) # trash documentation, no clue what this returns

    img = svo_image.get_data()
    depth_map = svo_depth.get_data()
    depth_map = np.transpose(depth_map)
    # depth_confidance_map = svo_confidance.get_data()
    point_cloud = svo_xyz.get_data()
    # print(f"point_cloud shape: {point_cloud.shape}")
    

    # get coords for all keypoints within the current frame
    # note: x,y coords are in pixels, depth is in meters
    for x, y in keypoints_2d[frame_idx, :]:

        depth = depth_map[int(x), int(y)]
        # b, g, r, a = img[int(x), int(y)]
        # depth_conf = depth_confidance_map[int(x), int(y)]
        xyz_point = point_cloud[int(x), int(y)]

        cv2.circle(img, (int(x), int(y)), 5, (0, 0, 100 * 255 / 100), 3)

        current_keypoints.append(np.array([x, y, depth]))
        current_xyz.append(xyz_point)
        

        
        # print(f"x: {x:.0f}, y: {y:.0f}, depth: {depth:.01f}, xy_conf: {xy_conf:.01f}, depth_conf: {depth_conf:.0f}")

    current_keypoints = np.array(current_keypoints)
    keypoints_3d.append(current_keypoints)
    keypoints_point_cloud.append(current_xyz)

#     ## show frame
    cv2.imshow("vid", cv2.resize(img, (800, 600)))

    if cv2.waitKey(25) & 0xFF == ord("q"):
        break


cv2.destroyAllWindows()
zed.close()

# shape: frames * kpts * (x, y, depth,xy_conf, depth_conf)
keypoints_3d = np.array(keypoints_3d)
print(f"keypoints 3d shape: {keypoints_3d.shape}")

keypoints_point_cloud = np.array(keypoints_point_cloud)
# print(f"keypoints_xyz shape: {keypoints_xyz.shape}")

In [ ]:
keypoints_point_cloud[0,5]

## Get camera position (R,t)

In [ ]:
from Util.util import Pattern, get_points
import glob

# input/output folders
# data_dir = "./chessboard_data"
data_dir = "./stereo_videos/validation_test"
output_dir = "./stereo_videos/validation_test"

with open("./Util/camera_params.json", "r") as f:
    intrinsic_params = json.load(f)

# describe real world calibration pattern parameteres
board_pattern = Pattern(6, 4, 0.150)


# serial numbers, file names and images
sns_fns_imgs = [
    (
        get_serial_number(file),
        file.split(".png")[0].split("\\")[1],
        cv2.imread(file, cv2.IMREAD_GRAYSCALE),
    )
    for file in glob.glob(os.path.join(data_dir, "*.png"))
]

extrinsics = []
camera_coords = []


for sn, fn, img in sns_fns_imgs:
    print(f"Processing {fn}...")
    img_points, obj_points = get_points(img, fn, output_dir, board_pattern)
    print(f"sn: {sn}")
    cameraMatrix = np.array(intrinsic_params[sn]["left_sensor"]["camera_matrix"])
    distCoeffs = np.array(intrinsic_params[sn]["left_sensor"]["distortion_coeff"])

    f_x = cameraMatrix[0, 0]
    f_y = cameraMatrix[1, 1]
    c_x = cameraMatrix[0, 2]
    c_y = cameraMatrix[1, 2]

    success, rvec, t = cv2.solvePnP(
        obj_points,
        img_points,
        cameraMatrix,
        distCoeffs,
        useExtrinsicGuess=False,
        flags=cv2.SOLVEPNP_ITERATIVE,
    )  # CV_P3P ,CV_EPNP
    # success, rvec, tvec, inliners = cv.solvePnPRansac(obj_points, img_points,cameraMatrix,distCoeffs)
    if success:
        print(f"{fn}... OK")
        # extrinsics.append((sn, fn, np.concatenate((rvec, tvec), 1).reshape(1, 6), rvec, tvec))
        R, jac = cv2.Rodrigues(rvec)

        Xw = -np.matrix(R).T * np.matrix(t)
        camera_coords.append(Xw)

        print(f"distance: {round(np.linalg.norm(Xw),2)} m")

    else:
        print(f"{fn}... FAILED")


## Transform 3D keypoints to world-coordinate system

In [ ]:
keypoints_3d_world_frame = []

for kpt in keypoints_3d.transpose(1,2,0):

    u = kpt[0,:]
    v = kpt[1,:]
    Zc = kpt[2,:]

    Xw = ((u-c_x) * Zc/f_x) - t[0]
    Yw = ((v-c_y) * Zc/f_y) - t[1]
    Zw = Zc - t[2]

    # multiply by -1 so axes go in the correct direction
    Yw = -Yw
    Zw = -Zw
    
    keypoints_3d_world_frame.append(np.array([Zw,Xw,Yw]))

    
keypoints_3d_world_frame = np.array(keypoints_3d_world_frame)
keypoints_3d_world_frame.shape 

## Save data

In [ ]:
save_path = "stereo_videos\\validation_test\\43916681.npz"

with open(save_path, "w") as f:
    np.savez(
        save_path,
        keypoints_2d=keypoints_2d,
        confidence_2d=confidence_2d,
        color=color,
        kpt_labels=kpt_labels,
        keypoints_3d=keypoints_3d_world_frame,
        R = R,
        t = t
    )

print(f"Saved 3D keypoints to {save_path}")

## Filtering +  Qualisys - Stereo comparison

In [ ]:
# load QUALISYS data
file_path = "stereo_videos\\validation_test\\qualisys_mate_walking.npz"
data = np.load(file_path, allow_pickle=True)
qualisys_data = data['pose_data']
qualisys_labels = data['kpt_labels']
print(f"Qualisys data keys: {list(data.keys())}")
print(f"Qualisys data shape: {qualisys_data.shape}")

In [ ]:
## load STEREO data
npz_load_path = "stereo_videos\\validation_test\\43916681.npz"
loaded_data = np.load(npz_load_path)

keypoints_2d = loaded_data["keypoints_2d"]
confidence_2d = loaded_data["confidence_2d"]
color = loaded_data["color"]
keypoints_3d = loaded_data["keypoints_3d"]
kpt_labels = list(loaded_data["kpt_labels"])
R = loaded_data["R"]
t = loaded_data["t"]

print(f"loaded data : {npz_load_path}\nwith keys and shapes:")
[f"{key}:    {loaded_data[key].shape}" for key in list(loaded_data.keys())]



In [ ]:
# stereo labels
for idx,label in enumerate(kpt_labels):
    print(idx, label)

In [ ]:
# Qualisys labels
for i,l in enumerate(data['kpt_labels']):
    print(i,l)

In [ ]:
# perform filtering, visualize results as a comparison between Qualisys and Stereo
from scipy.signal import butter, filtfilt

keypoints_3d_filtered = loaded_data["keypoints_3d"]


keypoint_name = "right_ankle"
contra_keypoint_name = "left_ankle"

# Qualisys
qualisys_idx = list(qualisys_labels).index(keypoint_name)
qualisys_data = filter_data(qualisys_data, 100, "lowpass", [7], 4, 0.25)

Qd = qualisys_data[qualisys_idx, 0, :] / 1000
Qx = qualisys_data[qualisys_idx, 2, :] / 1000
Qy = qualisys_data[qualisys_idx, 1, :] / 1000


# stereo
stereo_idx = kpt_labels.index(keypoint_name)
keypoints_3d_filtered = filter_data(keypoints_3d, 60, "lowpass", [2], 4, 0.25)

#TODO: try to figure out why bandpass makes it broken
# keypoints_3d_filtered[:, :2, :] = filter_data(copy.deepcopy(keypoints_3d_filtered), 60, "lowpass", [2], 4, 0.25)[:, :2, :]
# keypoints_3d_filtered[:, 2, :] = filter_data(copy.deepcopy(keypoints_3d_filtered), 60, "bandpass", [0.2, 2], 4, 0.25)[:, 2, :]


shift = 225
Sd = keypoints_3d_filtered[stereo_idx, 0, shift:]
Sx = keypoints_3d_filtered[stereo_idx, 1, shift:]
Sy = keypoints_3d_filtered[stereo_idx, 2, shift:]


## plot
tQ = np.linspace(0, len(Qd) / 100, len(Qd))
tS = np.linspace(0, len(Sx) / 60, len(Sx))
xticks = np.array([0, 11, 16, 18, 23, 25, 29, 31, 36, 38, 43, 45, 50, 52, 57, 59, 64, 66])  # manual annotation
plt.close("all")
fig, axs = plt.subplots(6, 1, figsize=(13, 8))


## Horizontal ax 1-------------------------------------------------------
# axs[0].plot(tQ, Qy, "m", label="Gold")
axs[0].plot(tQ[1:],np.diff(Qy),'m',label = 'Gold')
# axs[1].plot(tS, Sx, "m", label="Stereo")
axs[1].plot(tS[1:],np.diff(Sx),'m',label='Stereo')
# axs[1].plot(tS,keypoints_3d[stereo_idx,1,shift:],"g--",alpha=0.5,label="raw")

## Horizontal ax 2-------------------------------------------------------
# axs[2].plot(tQ, Qd, label="Gold")
axs[2].plot(tQ[1:],np.diff(Qd),label='Gold')
# axs[3].plot(tS, Sd, label="Stereo")
axs[3].plot(tS[1:],np.diff(Sd),label='Stereo')
# axs[3].plot(tS,keypoints_3d[stereo_idx,0,shift:],"g--",alpha=0.5,label="raw")

## Vertical ax ----------------------------------------------------------
# axs[4].plot(tQ, Qx, "r", label="Gold")
axs[4].plot(tQ[1:],np.diff(Qx),'r',label = 'Gold')
# axs[5].plot(tS, Sy, "r", label="Stereo")
axs[5].plot(tS[1:],np.diff(Sy),'r',label='Stereo')
# axs[5].plot(tS,keypoints_3d[stereo_idx,2,shift:],"g--",alpha=0.5,label="raw")


plt.suptitle(keypoint_name)
axs[0].set_title("X axis (horizontal)")
axs[2].set_title("Y axis (horizontal)")
axs[4].set_title("Z axis (vertical)")
for ax in axs:
    ax.legend(loc="upper left")
    ax.set_xlim([min(xticks), max(xticks)])
    ax.set_xticks(xticks - 3.3, xticks)
    ax.grid()

plt.tight_layout()

In [ ]:
save_path = "stereo_videos\\validation_test\\43916681.npz"

with open(save_path, "w") as f:
    np.savez(
        save_path,
        keypoints_2d=keypoints_2d,
        confidence_2d=confidence_2d,
        color=color,
        kpt_labels=kpt_labels,
        keypoints_3d=keypoints_3d,
        keypoints_3d_filtered=keypoints_3d_filtered,
        R = R,
        t = t
    )

print(f"Saved 3D keypoints to {save_path}")

# Gait analysis

## Load data

In [2]:
## load STEREO data
stereo_file_path = "stereo_videos\\validation_test\\43916681.npz"
loaded_stereo_data = np.load(stereo_file_path)

stereo_data = loaded_stereo_data["keypoints_3d_filtered"]
stereo_labels = list(loaded_stereo_data["kpt_labels"])

print(f"stereo data: {stereo_data.shape}")
print(f"stereo_labels: {len(stereo_labels)}")
for prop, val in stereo_analysis_properties.items():
    print(f"{prop:<13} {val:.2f}")

print("\n \n")

# load QUALISYS data
qualisys_file_path = "stereo_videos\\validation_test\\qualisys_mate_walking.npz"
loaded_qualisys_data = np.load(qualisys_file_path, allow_pickle=True)
# convert values to meters
qualisys_data = loaded_qualisys_data["pose_data"] / 1000
qualisys_labels = loaded_qualisys_data["kpt_labels"].tolist()

# filter data
qualisys_data = filter_data(
    qualisys_data,
    qualysis_analysis_properties["fps"],
    "lowpass",
    [qualysis_analysis_properties["filter_cutoff"]],
    qualysis_analysis_properties["filter_order"],
    qualysis_analysis_properties["max_gap"],
)

print(f"qualisys_data: {qualisys_data.shape}")
print(f"qualisys_labels: {len(qualisys_labels)}")
for prop, val in qualysis_analysis_properties.items():
    print(f"{prop:<13} {val:.2f}")

stereo data: (26, 3, 3908)
stereo_labels: 26
fps           60.00
max_gap       0.25
filter_cutoff 7.00
filter_order  4.00
heel_thr      0.50
toe_thr       0.80
stride_min    0.65
stride_max    1.80
swing_min     0.30
stance_min    0.30

 

qualisys_data: (23, 3, 6336)
qualisys_labels: 23
fps           100.00
max_gap       0.25
filter_cutoff 7.00
filter_order  4.00
heel_thr      0.50
toe_thr       0.80
stride_min    0.65
stride_max    1.80
swing_min     0.30
stance_min    0.30


## Get gait events, analyze result

### Qualisys

In [4]:
qualisys_events = get_gait_events(
    qualisys_data, qualisys_labels, qualysis_analysis_properties, qualisys_file_path
)
qualisys_results = gait_analysis(
    qualisys_data, qualisys_events, qualisys_labels, qualysis_analysis_properties
)
qualisys_result_table, qualisys_result_df = display_results(qualisys_results)

   Parameter      Statistic   all  back_straight  front_straight
0      stime       Mean [m]  1.48           1.47            1.49
1      stime         CV [%]  3.55           2.53            4.23
2      stime  Asymmetry [%]  2.85           3.62            2.40
3       slen       Mean [m]  1.22           1.23            1.22
4       slen         CV [%]  2.52           3.17            1.39
5       slen  Asymmetry [%]  1.31           1.91            0.96
6        vel       Mean [m]  0.83           0.84            0.82
7        vel         CV [%]  3.86           3.29            4.07
8        vel  Asymmetry [%]  4.07           5.49            3.15
9      swing       Mean [m]  0.51           0.51            0.51
10     swing         CV [%]  2.90           2.68            3.08
11     swing  Asymmetry [%]  3.68           3.65            3.68
12     dsupp       Mean [m]  0.47           0.46            0.48
13     dsupp         CV [%] 11.29           8.06           13.12
14     dsupp  Asymmetry [

In [ ]:
qualisys_result_table

In [ ]:
qualisys_results

### Stereo

In [5]:
stereo_events = get_gait_events(
    stereo_data, stereo_labels, stereo_analysis_properties, stereo_file_path
)
stereo_results = gait_analysis(
    stereo_data, stereo_events, stereo_labels, stereo_analysis_properties
)
stereo_result_table,stereo_result_df = display_results(stereo_results)

   Parameter      Statistic   all  back_straight  front_straight
0      stime       Mean [m]  1.48           1.48            1.48
1      stime         CV [%]  7.46           8.79            6.16
2      stime  Asymmetry [%]  6.75          10.67            3.61
3       slen       Mean [m]  1.27           1.29            1.26
4       slen         CV [%]  3.32           3.92            2.47
5       slen  Asymmetry [%]  1.04           0.52            2.18
6        vel       Mean [m]  0.86           0.87            0.86
7        vel         CV [%]  8.03          10.48            4.86
8        vel  Asymmetry [%]  5.66          11.15            1.06
9      swing       Mean [m]  0.62           0.66            0.59
10     swing         CV [%] 17.62          23.30            4.05
11     swing  Asymmetry [%] 12.82          23.74            2.00
12     dsupp       Mean [m]  0.27           0.23            0.30
13     dsupp         CV [%] 32.43          16.18           33.32
14     dsupp  Asymmetry [

In [13]:
stereo_result_table

,Parameter,Statistic,Perspective,Value
0,stime,Mean [m],all,1.48
18,stime,Mean [m],front_straight,1.48
36,stime,Mean [m],back_straight,1.48
1,stime,CV [%],all,7.46
19,stime,CV [%],front_straight,6.16
37,stime,CV [%],back_straight,8.79
2,stime,Asymmetry [%],all,6.75
20,stime,Asymmetry [%],front_straight,3.61
38,stime,Asymmetry [%],back_straight,10.67
3,slen,Mean [m],all,1.27


### Compare results

In [22]:
diff = qualisys_result_df[["all","front_straight","back_straight"]]-stereo_result_df[["all","front_straight","back_straight"]]
diff["Parameter"] = qualisys_result_df["Parameter"]
diff["Statistic"] = qualisys_result_df["Statistic"]
diff = diff[["Parameter", "Statistic", "all", "front_straight", "back_straight"]]
diff.round(3)

,Parameter,Statistic,all,front_straight,back_straight
0,stime,Mean [m],-0.01,0.01,-0.02
1,stime,CV [%],-3.90,-1.94,-6.26
2,stime,Asymmetry [%],-3.90,-1.21,-7.05
3,slen,Mean [m],-0.05,-0.05,-0.06
4,slen,CV [%],-0.80,-1.07,-0.75
5,slen,Asymmetry [%],0.27,-1.22,1.40
6,vel,Mean [m],-0.04,-0.04,-0.04
7,vel,CV [%],-4.17,-0.79,-7.19
8,vel,Asymmetry [%],-1.59,2.09,-5.66
9,swing,Mean [m],-0.12,-0.09,-0.15


# WP

## Gait analysis

In [ ]:
## load STEREO data
npz_load_path = "stereo_videos\\validation_test\\43916681.npz"
loaded_data = np.load(npz_load_path)

keypoints_2d = loaded_data["keypoints_2d"]
confidence_2d = loaded_data["confidence_2d"]
color = loaded_data["color"]
keypoints_3d = loaded_data["keypoints_3d"]
keypoints_3d_filtered = loaded_data["keypoints_3d_filtered"]

kpt_labels = list(loaded_data["kpt_labels"])
R = loaded_data["R"]
t = loaded_data["t"]

print(f"loaded data : {npz_load_path}\nwith keys and shapes:")
[f"{key}:    {loaded_data[key].shape}" for key in list(loaded_data.keys())]

stereo_labels = list(loaded_data["kpt_labels"])

In [ ]:
# load QUALISYS data
file_path = "stereo_videos\\validation_test\\qualisys_mate_walking.npz"
data = np.load(file_path, allow_pickle=True)
qualisys_data = data['pose_data']/1000
qualisys_labels = data['kpt_labels'].tolist()
print(f"Qualisys data keys: {list(data.keys())}")
print(f"Qualisys data shape: {qualisys_data.shape}")

qualisys_data = filter_data(qualisys_data,100,"lowpass",[7],4,qualysis_analysis_properties['max_gap'])

### Step detection

#### Segmentation & Perspective (straight/turn, front/back)

In [ ]:
# data = loaded_data["keypoints_3d_filtered"]
# fs = 60


data = qualisys_data
fs = 100
kpt_labels = qualisys_labels
## straight vs turning detection -----------------------------------------------------------------------------------------
# get the y coord of shoulders (for qualisys x-y plane is horizontal, y coord was forwards/backwards movement)
shift = 225
shoulder_R = data[kpt_labels.index("right_shoulder"), 1, shift:]
shoulder_L = data[kpt_labels.index("left_shoulder"), 1, shift:]

shoulder_diff = np.abs(np.diff(shoulder_R - shoulder_L))
shoulder_diff = shoulder_diff / np.nanmax(shoulder_diff)

# Interpolate NaNs in shoulder_diff
if np.any(np.isnan(shoulder_diff)):
    nans = np.isnan(shoulder_diff)
    not_nans = ~nans
    shoulder_diff[nans] = np.interp(
        np.flatnonzero(nans), np.flatnonzero(not_nans), shoulder_diff[not_nans]
    )

# min distance from peak to peak (2 seconds of straight movement) in samples
min_peak_distance = 2 * fs

peaks, peak_properties = find_peaks(shoulder_diff, height=0.3, distance=min_peak_distance)
widths, width_heights, left_ips, right_ips = peak_widths(
    x=shoulder_diff, peaks=peaks, rel_height=0.8
)

# create mask to indicate straight walking segments (straight=1, turn=0)
turn_mask = np.zeros_like(shoulder_R)
for left_base_idx, right_base_idx in zip((left_ips).astype(int), right_ips.astype(int)):
    turn_segment_length = right_base_idx - left_base_idx
    turn_mask[left_base_idx:right_base_idx] = 1


## perspective  detection -------------------------------------------------------------------------------------------------

# # confidence value of a non-symmetrical keypoint (eg nose)
# nose_confidence = loaded_data["confidence_2d"][shift:, kpt_labels.index("nose")]
# nose_confidence_thr = 0.8
# perspective_nose = np.where(nose_confidence > nose_confidence_thr, 1, 0)

# shoulder based direction
perspective_shoulder = np.where(shoulder_L > shoulder_R, 1, 0)

#TODO: combine perspective and segmentation mask?


## PLOTTING ----------------------------------------------------------------------------------------------------------------

tS = np.linspace(0, len(shoulder_R) / fs, len(shoulder_R))
xticks = np.array(
    [0, 11, 16, 18, 23, 25, 29, 31, 36, 38, 43, 45, 50, 52, 57, 59, 64, 66]
)  # manual annotation

plt.close("all")
fig, axs = plt.subplots(2, 1, figsize=(14, 4))


axs[0].plot(tS[1:], shoulder_diff, label="shoulder diff")
axs[0].plot(tS, perspective_shoulder, label="front, back")
axs[0].plot(left_ips / fs, width_heights, "o", label="turn start", markersize=4)
axs[0].plot(right_ips / fs, width_heights, "o", label="turn end", markersize=4)
axs[0].plot(tS, turn_mask, label="straight/turn")

axs[1].plot(tS, shoulder_L, label="left")
axs[1].plot(tS, shoulder_R, label="right")


# axs[2].plot(tS[1:], shoulder_diff, label="shoulder diff")
# # axs[2].plot(tS, perspective_nose, label="front, back")
# axs[2].plot(left_ips / 60, width_heights, "o", label="turn start", markersize=4)
# axs[2].plot(right_ips / 60, width_heights, "o", label="turn end", markersize=4)
# axs[2].plot(tS, valid_segments, label="straight/turn")

# axs[3].plot(tS, nose_confidence, label="nose conf.")
# axs[3].hlines(
#     y=0.8, xmin=0, xmax=len(nose_confidence), colors="r", linestyles="--", label="threshold"
# )


axs[0].set_title("Shoulder position based")
# axs[2].set_title("Nose confidence based")

for ax in axs:
    # ax.set_xlim([min(xticks), max(xticks)])
    # ax.set_xticks(xticks - 3.3, xticks)
    ax.legend(loc="upper left")
    ax.grid()

plt.tight_layout()

#### Get gait-events (Brüning-Ridge velocity thr)

In [ ]:
properties = gait_analysis_properties
velocity = 1
side = "right"

# data = loaded_data["keypoints_3d_filtered"]  # stereo
# kpt_labels = loaded_data["kpt_labels"].tolist()
# fs = 60

data = qualisys_data # qualisys
kpt_labels = qualisys_labels
fs = 100

turn_mask, perspective = get_turns_and_perspective(data, kpt_labels, fs, 2)

# fs = properties["fps"]
heel_thr = properties["heel_thr"] * velocity
# use ankle marker too (in case the other 2 are not visible, also better for stereo - no obstuction issues)
ankle_thr = properties["heel_thr"] * velocity
big_toe_thr = properties["toe_thr"] * velocity

# Extract trajectories
heel = data[kpt_labels.index(f"{side}_heel"), :, :]
big_toe = data[kpt_labels.index(f"{side}_big_toe"), :, :]
ankle = data[kpt_labels.index(f"{side}_ankle"), :, :]

# Compute 3D velocity magnitudes
heel_vel = np.linalg.norm(np.diff(heel, axis=1), axis=0) * fs
ankle_vel = np.linalg.norm(np.diff(ankle, axis=1), axis=0) * fs
big_toe_vel = np.linalg.norm(np.diff(big_toe, axis=1), axis=0) * fs

# Ground contact detection based on thresholds
ground_contact = (
    (heel_vel < heel_thr) | (ankle_vel < ankle_thr) | (big_toe_vel < big_toe_thr)
).astype(int)

# Remove too short ground contact periods
min_gc_duration = int(properties["stance_min"] * fs)
min_no_gc_duration = int(properties["swing_min"] * fs)

contact_diff = np.diff(
    np.pad(ground_contact, 1, "constant")
)  # NOTE:pad beginning and end of array with 0 for diff
starts = np.nonzero(contact_diff == 1)[0]
ends = np.nonzero(contact_diff == -1)[0]

for start, end in zip(starts, ends):
    if end - start < min_gc_duration:
        ground_contact[start:end] = 0

# Remove too short swing periods
contact_diff = np.diff(np.pad(ground_contact, 1, "constant"))
starts = np.nonzero(contact_diff == 1)[0]
ends = np.nonzero(contact_diff == -1)[0]

for start, end in zip(starts, ends):
    if end - start < min_no_gc_duration:
        ground_contact[start:end] = 1


# exclude gait events of turn segments
ground_contact_diff = np.pad(np.diff(ground_contact), 1, "constant")
ground_contact_diff_straight = np.where(turn_mask, 0, ground_contact_diff)

# Identify initial contacts (ICs) and final contacts (FCs)
ICs = np.nonzero(ground_contact_diff_straight == 1)[0] + 1
FCs = np.nonzero(ground_contact_diff_straight == -1)[0] + 1


# Compute walking velocity from stride lengths and durations
# also filter false positive gait events based on them
stride_lengths = []
stride_durations = []
bad_GE_indices = []

direction = []

# refine ICs & Fcs based on stride duration and perspective
for i in range(len(ICs) - 1):
    current_IC = ICs[i]
    next_IC = ICs[i + 1]

    # temporal difference between current & next IC event
    stride_duration = (next_IC - current_IC) / fs
    

    # check if gait events belong to same straight segment
    facing_same_way = perspective[current_IC] == perspective[next_IC]
    stride_duration_ok = properties["stride_min"] <= stride_duration <= properties["stride_max"]
    
    if not stride_duration_ok and facing_same_way:
        bad_GE_indices.append(i + 1)

    if stride_duration_ok and facing_same_way:
        # take the spatial difference of [KEYPOINT] between current & next IC event
        stride_length_heel = np.linalg.norm(heel[:, next_IC] - heel[:, current_IC])
        # stride_length_ankle = np.linalg.norm(ankle[:, next_IC] - ankle[:, current_IC])
        # stride_length_toe = np.linalg.norm(big_toe[:, next_IC] - big_toe[:, current_IC])

        # stride_lengths.append(np.array([stride_length_heel, stride_length_ankle, stride_length_toe]))
        stride_lengths.append(stride_length_heel)
        stride_durations.append(stride_duration)
        direction.append(perspective[current_IC])


ICs = np.delete(ICs, bad_GE_indices)
FCs = np.delete(FCs, bad_GE_indices)

# print(f"idx     current IC      next IC      dur       lenght")


    # print(f"{i}  {current_IC/fs :.2f} s   {next_IC/fs :.2f} s  {stride_duration:.2f} s   {stride_length_heel:.2f} m ")

stride_lengths = np.array(stride_lengths)
stride_durations = np.array(stride_durations)

velocity = np.nanmean(stride_lengths/stride_durations)

# print(f"ICs: {ICs.shape},  FCs: {FCs.shape}")

# print(f"idx   heel      ankle     toe     duration      heel-ankle diff")
# for event_idx, (heel_len, ankle_len, toe_len) in enumerate(stride_lengths):
#     print(
#         f"{event_idx :<6}{heel_len:<10.2f}{ankle_len:<10.2f}{toe_len:<10.2f}{stride_durations[event_idx]:<10.2f}{heel_len-ankle_len:<10.2f}{direction[event_idx]}"
#     )

#### velocity & gait event plots

In [ ]:
# PLOTTING velocity, perspective, phase
tS = np.linspace(0, len(ankle_vel) / 60, len(ankle_vel))
xticks = np.array([0, 11, 16, 18, 23, 25, 29, 31, 36, 38, 43, 45, 50, 52, 57, 59, 64, 66])  # manual annotation

# create array to visualize gait events after filtering out bad ones
gait_event_vis = np.zeros_like(ground_contact_diff_straight)
gait_event_vis[ICs] = 1
gait_event_vis[FCs] = -1


plt.close("all")
fig, axs = plt.subplots(5, 1, figsize=(14, 9))

axs[0].plot(tS,ankle_vel,label="velocity")
axs[0].plot(tS, ground_contact*ankle_thr,label="ground contact at thr")

axs[1].plot(tS,big_toe_vel,label="velocity")
axs[1].plot(tS, ground_contact*big_toe_thr,label="ground contact at thr")

axs[2].plot(tS,heel_vel,label="velocity")
axs[2].plot(tS, ground_contact*heel_thr,label="ground contact at thr")

axs[3].plot(tS, ground_contact_diff[1:],alpha=0.4, label="ground contact diff")
axs[3].plot(tS[1:], turn_mask[2:], "r--", label="turn mask")
axs[3].plot(tS, gait_event_vis[1:],"b",alpha=1,label="gc diff straight")
axs[3].plot(tS, ground_contact_diff_straight[1:],"r",alpha=0.4,label="removed GEs")


axs[4].plot(tS, perspective[1:], label="perspective")


axs[0].set_title("ankle velocity")
axs[1].set_title("toe velocity")
axs[2].set_title("heel velocity")
axs[3].set_title("ground contact diff (gait events)")
axs[0].set_yticks([0,3,ankle_thr])
axs[1].set_yticks([0,3,big_toe_thr])
axs[2].set_yticks([0,3,heel_thr])
axs[3].set_yticks([-1,1,],["FC", "IC"])
axs[3].set_yticks([-1,1,],["FC", "IC"])
axs[4].set_yticks([0,1],["back","front"])

for ax in axs:
    # ax.set_xlim([min(xticks), max(xticks)])
    # ax.set_xticks(xticks - 3.3, xticks)
    ax.legend(loc="upper left")
    ax.grid()
    # ax.plot(tS, ground_contact*2,"m")


plt.tight_layout()

#### get all gait events (left, right)

In [ ]:
data = loaded_data["keypoints_3d_filtered"]  # stereo
kpt_labels = loaded_data["kpt_labels"].tolist()
fs = 60
gait_analysis_properties = stereo_analysis_properties

# data = qualisys_data # qualisys
# kpt_labels = qualisys_labels
# fs = 100

turn_mask, perspective = get_turns_and_perspective(data, kpt_labels,fs,2)

events = {"left": {}, "right": {}}

for side in ["left", "right"]:
    _, _, velocity = get_gait_events_one_side(data,kpt_labels,side,gait_analysis_properties,1,turn_mask,perspective,False,file_path)
    ICs, FCs, _ = get_gait_events_one_side(data,kpt_labels,side,gait_analysis_properties,velocity,turn_mask,perspective,True,file_path)
    events[side]["ICs"] = ICs
    events[side]["FCs"] = FCs


segments = []
for persp, turn in zip(perspective,turn_mask):
    match persp, turn:
        # front, straight
        case np.True_, np.False_:
            segment_name = "front_straight"
        # back, straight
        case np.False_, np.False_:
            segment_name = "back_straight"
        # all turns
        case _,_:
            segment_name = "turn"
    
    segments.append(segment_name)


IC_events = [
    {"frame": frame, "side": side, "perspective": segments[frame]}
    for side in ["left", "right"]
    for frame in events[side]["ICs"]
]
FC_events = [
    {"frame": frame, "side": side, "perspective": segments[frame]}
    for side in ["left", "right"]
    for frame in events[side]["FCs"]
]
gait_events = {"IC": IC_events, "FC": FC_events}
print(len(IC_events), len(FC_events))

#### Gait analisys

In [ ]:
gait_analysis_properties_qualisys = {
    "fps": 100,
    "max_gap": 0.25,
    "filter_cutoff": 7,
    "filter_order": 4,
    "heel_thr": 0.5,
    "toe_thr": 0.8,
    "stride_min": 0.65,
    "stride_max": 1.8,
    "swing_min": 0.3,
    "stance_min": 0.3
}
result = gait_analysis(qualisys_data, gait_events, kpt_labels, gait_analysis_properties_qualisys)
display_results(result)

In [ ]:
gait_analysis_properties_stereo = {
    "fps": 60,
    "max_gap": 0.25,
    "filter_cutoff": 7,
    "filter_order": 4,
    "heel_thr": 0.5,
    "toe_thr": 0.8,
    "stride_min": 0.65,
    "stride_max": 1.8,
    "swing_min": 0.3,
    "stance_min": 0.3
}
result = gait_analysis(keypoints_3d_filtered, gait_events, stereo_labels, gait_analysis_properties_stereo)
display_results(result)

In [ ]:
data = qualisys_data
fs = 100


stride_min = gait_analysis_properties["stride_min"]
stride_max = gait_analysis_properties["stride_max"]

perspectives = ["all", np.True_, np.False_]
# metrics of interest (for each side)
moi = {
    metric: []
    for metric in [
        "stime",
        "slen",
        "vel",
        "swing",
        "dsupp",
        "bos",
    ]
}
# output data structure
metrics = {
    perspective: {
        "left": copy.deepcopy(moi),
        "right": copy.deepcopy(moi),
    }
    for perspective in perspectives
}

# Initial- and Final-contact gait events
ICs = gait_events["IC"]
FCs = gait_events["FC"]

# iterate Initial Contact gait events
for i, IC in enumerate(ICs):

    # ipsilateral and contralateral sides for current gait event
    ipsi, contra = IC["side"], "left" if IC["side"] == "right" else "right"

    # perspective value ('straight'/'turn') or 'all' if missing
    perspective = IC.get("perspective", "all")

    if perspective not in perspectives:
        continue

    # get the next ipsilateral event's global index (relative to the whole event list), return None otherwise
    same_foot_next_idx = next(
        (j for j, event in enumerate(ICs[i + 1 :], start=i + 1) if event["side"] == ipsi), None
    )

    # check if event order is correct, filter false positives
    if same_foot_next_idx is not None:

        # stride time = time elapsed between heelstrikes of the same foot
        stime = (ICs[same_foot_next_idx]["frame"] - IC["frame"]) / fs

        # stride time falls in realistic time range
        if stride_min <= stime <= stride_max:
            # frame index of current and next IC event (ipsilateral)
            IC0, IC2 = IC["frame"], ICs[same_foot_next_idx]["frame"]

            # frame index of first contralateral heel strike (between current and next ipsilateral)
            IC1 = get_frame_index(ICs, contra, IC0, IC2)

            FC0, FC1, FC2 = None, None, None

            # if there's a next contralateral GE
            if IC1:
                IC1 = IC1[0]

                # see gait_events.png
                FC0 = get_frame_index(FCs, contra, IC0, IC1)
                FC1 = get_frame_index(FCs, ipsi, IC1, IC2)
                FC2 = get_frame_index(FCs, ipsi, IC2, IC2 + int(fs * stride_max))

                FC0 = FC0[0] if FC0 else None
                FC1 = FC1[0] if FC1 else None
                FC2 = FC2[0] if FC2 else None

            # if either of the values is None, ignore it
            if any(x is None for x in [IC0, IC1, IC2, FC0, FC1, FC2]):
                continue

            # heel point
            HP0 = np.nanmedian(data[kpt_labels.index(f"{ipsi}_heel"), :, IC0:FC0], axis=1)
            HP2 = np.nanmedian(data[kpt_labels.index(f"{ipsi}_heel"), :, IC2:FC2], axis=1)
            HP1 = np.nanmedian(
                data[kpt_labels.index(f"{contra}_heel"), :, IC1:FC1], axis=1
            )

            # stride lenght on xy plane
            slen = np.linalg.norm(HP2[:2] - HP0[:2])
            vel = slen / stime
            bos = np.linalg.norm(np.cross(HP2 - HP1, HP1 - HP0)) / np.linalg.norm(HP2 - HP1)
            swing = (IC2 - FC1) / fs
            dsupp = ((FC0 - IC0) + (FC1 - IC1)) / fs

            for pers in ["all", perspective]:
                metrics[pers][ipsi]["stime"].append(stime)
                metrics[pers][ipsi]["slen"].append(slen)
                metrics[pers][ipsi]["vel"].append(vel)
                metrics[pers][ipsi]["swing"].append(swing)
                metrics[pers][ipsi]["dsupp"].append(dsupp)
                metrics[pers][ipsi]["bos"].append(bos)

parameters = {}
for state, data in metrics.items():
    parameters[state] = {
        metric: {
            **compute_pooled_stats(data["left"][metric], data["right"][metric]),
            "asymmetry": compute_asymmetry(data["left"][metric], data["right"][metric]),
        }
        for metric in data["left"]
    }
parameters

## PLY

In [ ]:
import open3d as o3d

In [ ]:
print(f"keypoints_3d shape: {keypoints_3d.shape} (kpt * dims * frames)")
print(f"color shape:        {color.shape} (frames * kpt * rgb)")

In [ ]:
keypoints_3d_one_frame = keypoints_3d[:,:,0] #(kpt * dims * frames)
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(keypoints_3d_one_frame)
pcd.colors = o3d.utility.Vector3dVector(color[0,:,:]/255.0)
o3d.io.write_point_cloud("./data.ply", pcd, write_ascii=True)
o3d.visualization.draw_geometries([pcd])


## Spectral anal + filt

In [ ]:
keypoints_3d = loaded_data["keypoints_3d_wf"]## TODO: remname keypoints_3d_wf to keypoints_3d
Sy = keypoints_3d[stereo_idx, 2, shift:]

nans = np.isnan(Sy)

# if there are nans, interpolate the missing values for subsequent filtering
if np.any(nans):
    valid_indices = ~nans
    Sy[nans] = np.interp(np.flatnonzero(nans), np.flatnonzero(valid_indices), Sy[valid_indices])

plt.close('all')
plt.figure(figsize=(15,2))
fft_Sy = fft(Sy)
N = len(fft_Sy)
n = np.arange(N)
freq = n / (N / 60)

PSD = fft_Sy * np.conj(fft_Sy) / N
PSD = np.array(PSD)
limit = 0.2
indices =  PSD < limit # indices 
PSDClean = PSD*indices 

# indices[0] = False

fhat = indices * fft_Sy
ffilt = np.fft.ifft(fhat)

b, a = butter(N=4, Wn=[2,3], btype="bandpass", analog=False, fs=60)
Sy_filt = filtfilt(b, a, ffilt)

plt.close('all')
fig, axs = plt.subplots(2, 1, figsize=(13, 4))
axs[0].plot(freq,PSD)
axs[0].set_xlim([0,6])
axs[0].set_ylim([0,5])
axs[0].hlines(limit,0,6, color='r', linestyles='dashed')


axs[1].plot(freq,PSDClean)
axs[1].set_xlim([0,6])

plt.show()



plt.figure(figsize=(15,4))
plt.plot(ffilt,'m')
plt.plot(Sy+.5,'b')
plt.plot(Sy_filt-.5,'r')

plt.show()

In [ ]:
indices[:10]

In [ ]:
plt.close('all')
plt.figure(figsize=(13,6))

bandpass = [0.2, 15]
# orders = [1, 4,7, 8]
# band_mins = [0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]
band_maxs = [4,5,8,10,15,20]
values = band_maxs
for idx, value in enumerate(values,1):
    b, a = butter(6, [0.2, value], fs=60, btype="band", analog=False)
    result = filtfilt(b,a,ry)
    plt.subplot(len(values),1,idx)
    plt.plot(result, label=str(value))
    plt.hlines(0,0,len(result),'r')
    plt.legend()
    

In [ ]:
# keypoints_over_time.transpose(1,2,0).shape, 
keypoints_3d.transpose(1,2,0).shape

In [ ]:
bandpass = [0.2, 4]
b, a = butter(6, bandpass, fs=60, btype="band", analog=False)

test = []

for kpt_idx, kpt in enumerate(raw_keypoints.transpose(1,2,0)):
    # fx = filtfilt(b, a, kpt[0,:])
    # fy = filtfilt(b, a, kpt[1,:])
    sx = np.multiply(kpt[0,:], scale)
    # fy = kpt[1,:]
    fy = filtfilt(b,a,kpt[1,:])
    sfy = np.multiply(fy, scale)

    depth = keypoints_3d.transpose(1,2,0)[kpt_idx,0,:]
    # print(fx.shape, fy.shape, depth.shape)
    test.append(np.stack((depth,sx,sfy)))
test= np.array(test)
print(test.shape)
type(test)
    

In [ ]:
faszom = "stereo_videos\\validation_test\\test.npz"
with open(faszom, "w") as f:
    np.savez(faszom, raw_keypoints = raw_keypoints, keypoints_2d=keypoints_2d, keypoints_3d=keypoints_3d, kpt_labels=kpt_labels, scale=scale, test = test)

print(f"Saved 3D keypoints to {faszom}")